# Train And Evaluate On Colab GPU

Launcher notebook for running EGNN or HGNN on Colab. All implementation code lives in the repo; this notebook only clones the repo, copies data from Drive, trains, evaluates, and saves artifacts back to Drive.

Before running:
1. Select a GPU runtime: `Runtime > Change runtime type > GPU`.
2. Put `train.h5`, `val.h5`, and `test.h5` under `MyDrive/masters-thesis/data/`.
3. Paste a GitHub token in the first cell if the repo is private.

Main choice:
- `MODEL = "hgnn"` for the Hamiltonian energy model.
- `MODEL = "egnn"` for the direct dynamics baseline.

Outputs are saved to `MyDrive/masters-thesis/runs/<model>/<run_id>/`.

In [ ]:
# Cell 1: Runtime settings, Drive mount, and repo clone
from pathlib import Path
import subprocess

from google.colab import drive

drive.mount("/content/drive")

GITHUB_TOKEN = ""  # @param {type:"string"}
REPO = "AlexNeagu123/msc-thesis-gnn-nbody"
DRIVE = Path("/content/drive/MyDrive/masters-thesis")

MODEL = "hgnn"  # @param ["hgnn", "egnn"]
EPOCHS = 500  # @param {type:"integer"}
RUN_TRAINING = True  # @param {type:"boolean"}
RUN_EVALUATION = True  # @param {type:"boolean"}

assert MODEL in {"egnn", "hgnn"}, MODEL
assert EPOCHS > 0, EPOCHS
if not GITHUB_TOKEN:
    raise ValueError("Paste a GitHub token before cloning the private repo.")

repo_dir = Path("/content/repo")
clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"

if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", clone_url, str(repo_dir)], check=True)

%cd /content/repo
print(f"Selected model: {MODEL}")
print(f"Epochs: {EPOCHS}")

In [ ]:
# Cell 2: Install dependencies and verify GPU
!pip install -q -r requirements.txt

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Select a GPU runtime before continuing.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# Cell 3: Copy data from Drive and verify shapes
import h5py

data_dir = Path("data/output")
data_dir.mkdir(parents=True, exist_ok=True)

for name in ["train.h5", "val.h5", "test.h5"]:
    src = DRIVE / "data" / name
    dst = data_dir / name
    if not src.exists():
        raise FileNotFoundError(f"Missing Drive data file: {src}")
    subprocess.run(["cp", str(src), str(dst)], check=True)

for name in ["train.h5", "val.h5", "test.h5"]:
    with h5py.File(data_dir / name, "r") as f:
        print(f"{name}: {f['trajectories'].shape}")

In [ ]:
# Cell 4: Patch the selected config for this Colab run
import yaml

cfg_path = Path(f"configs/{MODEL}.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["training"]["epochs"] = int(EPOCHS)
cfg["training"]["device"] = "auto"
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

print(cfg_path)
print(cfg_path.read_text())

In [ ]:
# Cell 5: Train the selected model
if RUN_TRAINING:
    subprocess.run(
        ["python", "-m", "training.train", "--config", f"configs/{MODEL}.yaml"],
        check=True,
    )
else:
    print("Training skipped.")

checkpoint_root = Path("checkpoints") / MODEL
run_ids = sorted(path.name for path in checkpoint_root.iterdir() if path.is_dir())
if not run_ids:
    raise RuntimeError(f"No checkpoint runs found under {checkpoint_root}.")

RUN_ID = run_ids[-1]
BEST_CHECKPOINT = checkpoint_root / RUN_ID / "best.pt"
print(f"Run id: {RUN_ID}")
print(f"Best checkpoint: {BEST_CHECKPOINT}")

In [ ]:
# Cell 6: Run official numeric evaluation
if RUN_EVALUATION:
    subprocess.run(
        [
            "python",
            "-m",
            "evaluation.evaluate",
            "--config",
            f"configs/{MODEL}.yaml",
            "--checkpoint",
            str(BEST_CHECKPOINT),
            "--device",
            "cuda",
        ],
        check=True,
    )
else:
    print("Evaluation skipped.")

EVAL_DIR = Path("results/evaluation") / MODEL / RUN_ID
print(f"Evaluation directory: {EVAL_DIR}")
if EVAL_DIR.exists():
    print((EVAL_DIR / "summary.csv").read_text())

In [ ]:
# Cell 7: Save checkpoints, logs, and evaluation outputs to Drive
import shutil

save_dir = DRIVE / "runs" / MODEL / RUN_ID
save_dir.mkdir(parents=True, exist_ok=True)

items = [
    (Path("checkpoints") / MODEL / RUN_ID, save_dir / "checkpoint"),
    (Path("logs") / MODEL / RUN_ID, save_dir / "logs"),
    (Path("results/evaluation") / MODEL / RUN_ID, save_dir / "evaluation"),
]

for src, dst in items:
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"saved {src} -> {dst}")
    else:
        print(f"missing, skipped: {src}")

print(f"Drive output: {save_dir}")